In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
# Для реального использования, раскомментируйте строку ниже
# html_content = requests.get('http://duma.gov.ru/duma/deputies/1/').text

# Используем HTML, который вы предоставили (для примера)
html_content = """<здесь должен быть ваш огромный HTML, но для краткости я его опускаю>"""
# В вашем реальном коде вы бы использовали: html_content = response.text

# Для демонстрации, предположим, что у нас есть переменная с HTML
# Я сохраню ваш HTML во временный файл или буду использовать его напрямую в парсере.

# Распарсим HTML
soup = BeautifulSoup(html_content, 'html.parser')

# Найдем все блоки депутатов
deputy_blocks = soup.find_all('li', class_='list-persons__item')

# Создадим списки для хранения данных
data = {
    'Фамилия': [],
    'Имя_Отчество': [],
    'Полное_имя': [],
    'Ссылка_на_фото': [],
    'Информация': []
}

# Пройдемся по каждому блоку и извлечем информацию
for block in deputy_blocks:
    # Ищем блок с персональными данными
    person_block = block.find('div', class_='person')
    if not person_block:
        continue

    # Извлекаем фамилию и имя с отчеством
    title_block = person_block.find('h2', class_='person__title--s')
    if title_block:
        surname_tag = title_block.find('strong')
        surname = surname_tag.get_text(strip=True) if surname_tag else ''
        
        # Имя и отчество
        name_part_tag = title_block.find('span', class_='second-name')
        name_part = name_part_tag.get_text(strip=True) if name_part_tag else ''
        
        full_name = f"{surname} {name_part}".strip()
    else:
        surname = ''
        name_part = ''
        full_name = ''

    # Извлекаем ссылку на фото (если есть)
    img_tag = person_block.find('img', itemprop='image')
    img_src = img_tag.get('src') if img_tag else ''
    # Формируем полную ссылку
    if img_src and not img_src.startswith('http'):
        img_src = 'http://duma.gov.ru' + img_src

    # Извлекаем информацию из блока person__post
    info_block = person_block.find('div', class_='person__post')
    info_text = info_block.get_text(' ', strip=True) if info_block else ''

    # Добавляем данные в словарь
    data['Фамилия'].append(surname)
    data['Имя_Отчество'].append(name_part)
    data['Полное_имя'].append(full_name)
    data['Ссылка_на_фото'].append(img_src)
    data['Информация'].append(info_text)

# Создаем DataFrame
df = pd.DataFrame(data)

# Выводим первые несколько строк для проверки
print(df.head())

# Сохраняем в CSV, если нужно
# df.to_csv('deputies_duma.csv', index=False, encoding='utf-8-sig')